# **02477 Exam Code Reference**

One runnable block per scenario. Change the **# ── INPUTS ──** section, run, read the answer.

**How to use:** Ctrl+F the section name → update numbers → Shift+Enter → copy to answer sheet.

---
## Contents
1. Imports & Helpers
2. Linear Gaussian Systems
3. Bayesian Linear Regression
4. GP Regression
5. GP Classification (Laplace)
6. Binary Classification (Logistic + Laplace)
7. Multi-class Classification (Softmax)
8. Grid Approximation
9. Mixture Models
10. Variational Inference (Mean-Field)
11. Metropolis MCMC
12. Monte Carlo Estimators
13. Decision Theory
14. Credibility Intervals & Tail Probabilities
15. Ancestral Sampling

---
## 1. Imports & Helpers

Run this cell first — every section below depends on it.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, multivariate_normal as mvn, poisson
np.random.seed(42)

# Reusable helpers
sigmoid  = lambda a: 1.0 / (1.0 + np.exp(-a))
softmax  = lambda f: np.exp(f - np.max(f)) / np.sum(np.exp(f - np.max(f)))
log_npdf = lambda x, m, v: -0.5*np.log(2*np.pi*v) - 0.5*(x-m)**2/v
npdf     = lambda x, m, v: np.exp(log_npdf(x, m, v))
log_ber  = lambda y, p: y*np.log(np.clip(p,1e-12,1)) + (1-y)*np.log(np.clip(1-p,1e-12,1))
entropy  = lambda p: -np.sum(p[p>0] * np.log(p[p>0]))
print("Helpers loaded.")


---
## 2. Linear Gaussian Systems

### 2a — Reverse conditional $p(x|z) = \mathcal{N}(x|m, S)$

System: $z = Ax + b + n_1$, $x\sim\mathcal{N}(\mu_x,\Sigma_x)$, $n_1\sim\mathcal{N}(0,\Sigma_1)$

Formulas: $S^{-1} = \Sigma_x^{-1} + A^T\Sigma_1^{-1}A$, $\quad m = S[A^T\Sigma_1^{-1}(z-b) + \Sigma_x^{-1}\mu_x]$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
A       = np.array([[0.,1.],[1.,0.]])   # linear map in z = Ax + b + n1
b       = np.array([1., 1.])            # offset
Sigma1  = 0.5 * np.eye(2)              # noise cov of z  (1/2 * I in exam)
Sigma_x = np.eye(2)                    # prior cov of x
mu_x    = np.zeros(2)                  # prior mean of x
z_obs   = np.array([2., 3.])           # observed z value

# ── CALCULATION ──────────────────────────────────────────────────────────────
S_inv = np.linalg.inv(Sigma_x) + A.T @ np.linalg.inv(Sigma1) @ A
S     = np.linalg.inv(S_inv)
m     = S @ (A.T @ np.linalg.inv(Sigma1) @ (z_obs - b)
             + np.linalg.inv(Sigma_x) @ mu_x)

print("p(x|z) = N(x | m, S)")
print(f"  Posterior precision S_inv = {np.round(S_inv,4)}")
print(f"  Posterior covariance S    = {np.round(S,4)}")
print(f"  Posterior mean m          = {np.round(m,4)}")


### 2b — Marginal $p(y)$ through chain $x \to z \to y$

System: $z = Ax+b+n_1$, $y = Wz+n_2$. Uses marginalisation formula: $p(y)=\mathcal{N}(y|W\mu_z,\Sigma_2+W\Sigma_z W^T)$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
A2       = np.array([[0.,1.],[1.,0.]])   # x -> z
b2       = np.array([1., 1.])
Sigma_x2 = np.eye(2)                    # prior cov of x
mu_x2    = np.zeros(2)                  # prior mean of x
Sigma1b  = 0.5 * np.eye(2)             # noise cov of z
W2       = np.array([[1.,1.],[1.,0.]]) # z -> y
Sigma2   = 0.5 * np.eye(2)             # noise cov of y

# ── CALCULATION ──────────────────────────────────────────────────────────────
# p(z): marginalise x
mu_z    = A2 @ mu_x2 + b2
Sigma_z = Sigma1b + A2 @ Sigma_x2 @ A2.T

# p(y): marginalise z
mu_y    = W2 @ mu_z
Sigma_y = Sigma2 + W2 @ Sigma_z @ W2.T

print("p(z) = N(z | mu_z, Sigma_z)")
print(f"  mu_z    = {np.round(mu_z, 4)}")
print(f"  Sigma_z = {np.round(Sigma_z, 4)}")
print()
print("p(y) = N(y | mu_y, Sigma_y)")
print(f"  mu_y    = {np.round(mu_y, 4)}")
print(f"  Sigma_y = {np.round(Sigma_y, 4)}")


---
## 3. Bayesian Linear Regression

$y_n = w^T\phi(x_n)+\epsilon_n$, prior $w\sim\mathcal{N}(0,\alpha^{-1}I)$, noise $\epsilon_n\sim\mathcal{N}(0,\sigma^2)$

Formulas: $S=(\alpha I+\frac{1}{\sigma^2}\Phi^T\Phi)^{-1}$, $\;m=\frac{1}{\sigma^2}S\Phi^Ty$, $\;p(y^*|y,x^*)=\mathcal{N}(y^*|m^T\phi^*,\phi^{*T}S\phi^*+\sigma^2)$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
x_data = np.array([0., 1., 2., 4., 5.])
y_data = np.array([1., 0.5, -0.1, -0.9, 1.1])
alpha  = 1/2           # prior precision  (prior variance = 1/alpha = 2)
sigma  = 1/5           # noise std dev
x_star = 3.0           # prediction point

# Feature map — change to match your exam (linear, polynomial, etc.)
phi    = lambda x: np.array([1, x, x**2, x**3])   # degree-3 polynomial
Phi    = np.vstack([phi(xi) for xi in x_data])      # N x D design matrix

# ── CALCULATION ──────────────────────────────────────────────────────────────
D      = Phi.shape[1]
w_MLE  = np.linalg.solve(Phi.T @ Phi, Phi.T @ y_data)
S_inv  = alpha*np.eye(D) + (1/sigma**2)*Phi.T@Phi
S_post = np.linalg.inv(S_inv)
m_post = (1/sigma**2) * S_post @ Phi.T @ y_data

phi_s     = phi(x_star)
pred_mean = m_post @ phi_s
epi_var   = phi_s @ S_post @ phi_s
pred_var  = epi_var + sigma**2

print(f"w_MLE  = {np.round(w_MLE, 2)}")
print(f"m_post = {np.round(m_post, 2)}")
print()
print(f"p(f*|y, x*={x_star}) = N(f*| {pred_mean:.4f}, {epi_var:.4f})")
print(f"p(y*|y, x*={x_star}) = N(y*| {pred_mean:.4f}, {pred_var:.4f})")
print(f"  Epistemic var (from w) = {epi_var:.4f}")
print(f"  Aleatoric var (noise)  = {sigma**2:.4f}")


---
## 4. GP Regression

$f\sim\mathcal{GP}(0,k_{\text{SE}})$, $y_n=f(x_n)+\epsilon_n$, $\epsilon_n\sim\mathcal{N}(0,\sigma^2)$

Formulas: $\mu_{f^*}=k_*(K+\sigma^2I)^{-1}y$, $\;\sigma^2_{f^*}=k_{**}-k_*(K+\sigma^2I)^{-1}k_*^T$, $\;\sigma^2_{y^*}=\sigma^2_{f^*}+\sigma^2$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
x_gp     = np.array([-2.17, 1.99, 0.57, -3.01, -1.16, 3.30, -4.85, -0.86])
y_gp     = np.array([ 0.88, 0.46,-0.06,  0.98,  0.45, 0.88, -0.66,  0.05])
kappa    = 0.7               # kernel amplitude (kappa^2 = prior variance)
ell      = 0.5*np.sqrt(2)   # lengthscale
sigma_gp = 0.2              # noise std dev
x_star_gp= 1.0              # prediction point

# ── CALCULATION ──────────────────────────────────────────────────────────────
N_gp = len(x_gp)
K_gp = kappa**2 * np.exp(-0.5*(x_gp[:,None]-x_gp[None,:])**2/ell**2) + 1e-8*np.eye(N_gp)
C_gp = K_gp + sigma_gp**2 * np.eye(N_gp)     # C = K + sigma^2 * I

k_s  = kappa**2 * np.exp(-0.5*(x_star_gp - x_gp)**2/ell**2)   # cross-cov (N,)
k_ss = kappa**2                                                  # k(x*,x*) = kappa^2

# Posterior of f* and y*
mu_f   = k_s @ np.linalg.solve(C_gp, y_gp)
var_f  = k_ss - k_s @ np.linalg.solve(C_gp, k_s)
var_y  = var_f + sigma_gp**2

# Log marginal likelihood via Cholesky
L_gp   = np.linalg.cholesky(C_gp)
v_gp   = np.linalg.solve(L_gp, y_gp)
log_ml = -0.5*N_gp*np.log(2*np.pi) - np.sum(np.log(np.diag(L_gp))) - 0.5*np.dot(v_gp,v_gp)

print(f"Prior predictive: p(y*|x*={x_star_gp}) = N(y*| 0, {k_ss+sigma_gp**2:.4f})")
print(f"  k(x*,x*) = kappa^2 = {k_ss:.4f},  sigma^2 = {sigma_gp**2:.4f}")
print()
print(f"Posterior of f*:  N(f*| {mu_f:.4f}, {var_f:.4f})")
print(f"Posterior of y*:  N(y*| {mu_f:.4f}, {var_y:.4f})")
print(f"Log marginal likelihood = {log_ml:.2f}")


### 4b — GP Posterior at training points, and custom kernel covariance

In [ ]:
# ── GP posterior at ALL training points p(f|y,x) ─────────────────────────
post_mean_f = K_gp @ np.linalg.solve(C_gp, y_gp)
post_cov_f  = K_gp - K_gp @ np.linalg.solve(C_gp, K_gp)

print("GP Posterior p(f|y,x) = N(f | m_post, K_post)")
print(f"  m_post (posterior mean at training points):  {np.round(post_mean_f,2)}")
print(f"  Posterior variances (diagonal of K_post):    {np.round(np.diag(post_cov_f),4)}")
print()

# ── Custom kernel covariance vector (e.g. k2 from 2024 exam) ─────────────
c1, c2  = 1.0, 1.0
ell2    = 1.0/np.sqrt(2)
xstar_k = -1.0

def k2(xi, xs, c1, c2, ell):
    return c1*(1 + np.abs(xi-xs)/(2*ell**2))**(-1) + c2*xi*xs

cov_k2 = np.array([k2(xi, xstar_k, c1, c2, ell2) for xi in x_gp])
print(f"Prior covariance cov(f, f*) for x*={xstar_k} using k2:")
print(np.round(cov_k2, 2))


---
## 5. GP Classification — Laplace Posterior Predictive

$y_n|f_n\sim\text{Ber}(\sigma(f_n))$. Given Laplace approximation $q(f)=\mathcal{N}(f|m,S)$.

Formulas: $\mu_{f^*}=k^TK^{-1}m$, $\;\sigma^2_{f^*}=k_{**}-k^TK^{-1}(K-S)K^{-1}k$, $\;p(y^*=1)\approx\Phi(\mu_{f^*}/\sqrt{8/\pi+\sigma^2_{f^*}})$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
x_gpc   = np.array([[1.],[2.],[3.]])    # training inputs (N x 1)
x_star_c= np.array([[4.]])             # test point (1 x 1)

# Laplace approximation (given on exam)
m_lap = np.array([1.56, 0.84, -0.32])
S_lap = np.array([[2.69,1.46,0.42],[1.46,1.78,1.2],[0.42,1.2,2.13]])

# Kernel — change to your exam kernel
def kernel_gpc(x1, x2):
    dist_sq = (x1 - x2.T)**2
    return 5.0 * (1.0 + np.exp(-0.25 * dist_sq))

# ── CALCULATION ──────────────────────────────────────────────────────────────
K_c   = kernel_gpc(x_gpc, x_gpc) + 1e-8*np.eye(len(x_gpc))
k_vec = kernel_gpc(x_star_c, x_gpc).T    # (N,1)
k_ss_c= kernel_gpc(x_star_c, x_star_c)[0,0]

Kinv_k     = np.linalg.solve(K_c, k_vec)
mu_fstar_c = (k_vec.T @ np.linalg.solve(K_c, m_lap))[0]
var_fstar_c= k_ss_c - (Kinv_k.T @ (K_c - S_lap) @ Kinv_k)[0,0]
probit_arg  = mu_fstar_c / np.sqrt(8/np.pi + var_fstar_c)
p_y1_c      = norm.cdf(probit_arg)

print(f"Prior variance k(x*,x*) = {k_ss_c:.4f}")
print(f"Prior predictive p(y*=0|x*) = 0.5  (zero-mean GP is symmetric)")
print()
print(f"Posterior of f*:  N(f*| {mu_fstar_c:.4f}, {var_fstar_c:.4f})")
print(f"Probit approx:    p(y*=1|y,x*) = Phi({probit_arg:.4f}) = {p_y1_c:.4f}")


---
## 6. Binary Classification — MAP + Laplace

$y_n|w,x_n\sim\text{Ber}(\sigma(w^Tx_n))$, $w\sim\mathcal{N}(0,\lambda^{-1}I)$

Hessian: $\mathcal{H}=-X^TSX-\lambda I$ where $S_{nn}=\sigma(f_n)(1-\sigma(f_n))$. Laplace: $S=-\mathcal{H}^{-1}$.

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
X_bc     = np.array([[0.,0.],[1.,1.]])  # design matrix rows are x_n
y_bc     = np.array([0., 1.])
lam_bc   = 1/4                          # prior precision
w_MAP_bc = 0.74077439*np.array([1.,1.]) # given MAP (or compute via scipy.optimize)
x_star_bc= np.array([0., 1.])          # test input

# ── CALCULATION ──────────────────────────────────────────────────────────────
# Plug-in prediction
p_plugin_bc = sigmoid(w_MAP_bc @ x_star_bc)
print(f"Plug-in: p(y*=1|y,x*) = sigma({w_MAP_bc @ x_star_bc:.4f}) = {p_plugin_bc:.4f}")

# Log joint at w_MAP
p_pred_bc   = sigmoid(X_bc @ w_MAP_bc)
log_p_bc    = (np.sum(log_npdf(w_MAP_bc, 0, 1/lam_bc))
               + np.sum(log_ber(y_bc, p_pred_bc)))
print(f"log p(y, w_MAP) = {log_p_bc:.4f}")

# Hessian
diag_S_bc    = p_pred_bc * (1 - p_pred_bc)
H_bc         = -X_bc.T @ np.diag(diag_S_bc) @ X_bc - lam_bc*np.eye(X_bc.shape[1])
S_laplace_bc = -np.linalg.inv(H_bc)
print(f"Hessian H  = {np.round(H_bc,4)}")
print(f"Laplace S  = {np.round(S_laplace_bc,4)}")

# Posterior of f* and probit
mu_f_bc  = w_MAP_bc @ x_star_bc
var_f_bc = x_star_bc @ S_laplace_bc @ x_star_bc
p_ystar_bc = norm.cdf(mu_f_bc / np.sqrt(8/np.pi + var_f_bc))
print(f"p(f*|y,x*) = N(f*| {mu_f_bc:.4f}, {var_f_bc:.4f})")
print(f"Probit approx: p(y*=1|y,x*) = {p_ystar_bc:.4f}")


---
## 7. Multi-class Classification — Softmax

$y_n|W,x_n\sim\text{Cat}(\text{softmax}(W\phi(x_n)))$, $W_{ij}\sim\mathcal{N}(0,\alpha^{-1})$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
W_MAP_mc  = np.array([[-0.5,-2.0],[3.0,0.0],[1.0,1.0]])  # K x D weight matrix
phi_mc    = lambda x: np.array([1., x])                   # feature map
x_star_mc = -1.0                                          # test point
# Posterior weight samples (from variational approx or MCMC)
W_samples_mc = [
    np.array([[-0.15,-1.92],[3.2,0.45],[1.37,0.80]]),
    np.array([[-0.31,-2.03],[2.98,0.08],[1.03,1.29]]),
    np.array([[-0.35,-1.98],[3.09,0.07],[1.30,0.96]]),
]
# Predictive distribution to analyse
p_dist_mc = np.array([0.00, 0.27, 0.73])

# ── CALCULATION ──────────────────────────────────────────────────────────────
phi_s_mc    = phi_mc(x_star_mc)
p_plugin_mc = softmax(W_MAP_mc @ phi_s_mc)
p_mc_mean   = np.mean([softmax(Wi @ phi_s_mc) for Wi in W_samples_mc], axis=0)

print("Plug-in (MAP) prediction:")
for k,p in enumerate(p_plugin_mc): print(f"  p(y*={k+1}) = {p:.4f}")

print("MC estimate (average over samples):")
for k,p in enumerate(p_mc_mean): print(f"  p(y*={k+1}) = {p:.4f}")

print(f"For p = {p_dist_mc}:")
print(f"  Confidence = {np.max(p_dist_mc):.4f}")
print(f"  Entropy    = {entropy(p_dist_mc):.4f}")


---
## 8. Grid Approximation

Discrete posterior $q(w_1,w_2)$ read from the exam figure as $(w_1, w_2, \text{prob})$ triples.

Formulas: $\mathbb{E}[w_1]=\sum w_1 q(w_1,w_2)$, $\;\mathbb{V}[w_1]=\sum(w_1-\mathbb{E}[w_1])^2 q$, $\;p(y^*|y,x^*)\approx\sum q(w)\mathcal{N}(y^*|f(x^*,w),\sigma^2)$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
# Read from the grid table in the exam figure: (w1, w2, probability)
grid = np.array([
    (0.5, 0.2, 0.12), (0.5, 0.3, 0.09),
    (0.6, 0.1, 0.12), (0.6, 0.2, 0.20), (0.6, 0.3, 0.12),
    (0.7, 0.1, 0.23), (0.7, 0.2, 0.12),
])
sigma2_grid  = 0.5    # likelihood noise variance
y_star_grid  = 4.0    # observed y*
x_star_grid  = 4.0    # test input x*

# Model function — change to your exam
f_grid = lambda w1, w2, x: np.exp(w1 + w2*x)

# ── CALCULATION ──────────────────────────────────────────────────────────────
w1s, w2s, probs = grid[:,0], grid[:,1], grid[:,2]
print(f"Probabilities sum to {probs.sum():.4f}")

E_w1  = np.sum(w1s * probs);  V_w1 = np.sum((w1s-E_w1)**2 * probs)
E_w2  = np.sum(w2s * probs);  V_w2 = np.sum((w2s-E_w2)**2 * probs)
print(f"E[w1] = {E_w1:.4f},  Var[w1] = {V_w1:.4f},  Std[w1] = {np.sqrt(V_w1):.4f}")
print(f"E[w2] = {E_w2:.4f},  Var[w2] = {V_w2:.4f}")

post_pred = np.sum([npdf(y_star_grid, f_grid(w1,w2,x_star_grid), sigma2_grid)*p
                    for w1,w2,p in grid])
print(f"p(y*={y_star_grid}|y, x*={x_star_grid}) = {post_pred:.4f}")


---
## 9. Mixture Models

### 9a — Mixture prior: $p(\theta)=\frac{1}{2}\mathcal{N}(\theta|-m,\tau^2I)+\frac{1}{2}\mathcal{N}(\theta|m,\tau^2I)$

Marginal likelihood uses $\int\mathcal{N}(y|X\theta,\sigma^2I)\mathcal{N}(\theta|\mu_0,\tau^2I)d\theta=\mathcal{N}(y|X\mu_0,\tau^2XX^T+\sigma^2I)$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
m_mix    = np.array([1., 1.])           # mixture centre
tau2_mix = 1.0                          # prior variance
X_mix    = np.array([[1.,0.5],[-1.,1.]])# design matrix
y_mix    = np.array([1., 0.])           # observations
sigma2_mix = 1.0                        # likelihood noise variance
theta_ev = np.array([0., 0.])          # point to evaluate

# ── CALCULATION ──────────────────────────────────────────────────────────────
D_mix = len(m_mix)
prior_mix = lambda t: (0.5*mvn.pdf(t,-m_mix,tau2_mix*np.eye(D_mix))
                     + 0.5*mvn.pdf(t, m_mix,tau2_mix*np.eye(D_mix)))
C_mix    = tau2_mix*X_mix@X_mix.T + sigma2_mix*np.eye(len(y_mix))
evidence = 0.5*mvn.pdf(y_mix,-X_mix@m_mix,C_mix) + 0.5*mvn.pdf(y_mix,X_mix@m_mix,C_mix)

p_prior = prior_mix(theta_ev)
p_lik   = mvn.pdf(y_mix, X_mix@theta_ev, sigma2_mix*np.eye(len(y_mix)))
p_post  = p_lik * p_prior / evidence

print(f"Prior p(theta={theta_ev})         = {p_prior:.4f}")
print(f"Likelihood p(y|theta={theta_ev}) = {p_lik:.4f}")
print(f"Marginal likelihood p(y)          = {evidence:.4f}")
print(f"Posterior p(theta={theta_ev}|y)  = {p_post:.4f}")
print(f"  (Bayes rule: {p_lik:.4f} * {p_prior:.4f} / {evidence:.4f})")


### 9b — Discrete switch $s\in\{0,1\}$: $p(y|v)$, $p(y)$, $p(s=1|y)$

Marginalise $s$ by sum rule, marginalise $v$ by linear Gaussian formula.

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
sigma2_s = 1.0    # base noise (s=0 component variance)
tau2_s   = 2.0    # extra variance when s=1
p_s1     = 1/4    # P(s=1)
mu_v     = 1.0    # prior mean of v
var_v    = 1.0    # prior variance of v
y_eval   = 1.5    # y value to evaluate

# ── CALCULATION ──────────────────────────────────────────────────────────────
# p(y|v) = (1-p)*N(y|v, sigma2) + p*N(y|v, sigma2+tau2)
# After integrating out v: marginal variances add
var_marg_s0 = sigma2_s + var_v
var_marg_s1 = sigma2_s + tau2_s + var_v

py_s0 = norm.pdf(y_eval, mu_v, np.sqrt(var_marg_s0))
py_s1 = norm.pdf(y_eval, mu_v, np.sqrt(var_marg_s1))
py    = (1-p_s1)*py_s0 + p_s1*py_s1
ps1_given_y = p_s1 * py_s1 / py

print(f"p(y|v) = {1-p_s1:.2f}*N(y|v,{sigma2_s}) + {p_s1:.2f}*N(y|v,{sigma2_s+tau2_s})")
print(f"p(y)   = {1-p_s1:.2f}*N(y|{mu_v},{var_marg_s0}) + {p_s1:.2f}*N(y|{mu_v},{var_marg_s1})")
print(f"       = {py:.4f}  at y={y_eval}")
print()
print(f"p(s=1|y={y_eval}) = {p_s1:.2f}*N({y_eval}|{mu_v},{var_marg_s1:.1f}) / p(y)")
print(f"                  = {p_s1*py_s1:.4f} / {py:.4f} = {ps1_given_y:.4f}")


---
## 10. Variational Inference — Mean-Field Gaussian

Entropy: $H[q]=\frac{1}{2}\sum_i\ln(2\pi e v_i)$. Intervals: $m_i\pm z\sqrt{v_i}$ (note: $v_i$ is variance, take $\sqrt{}$ for std).

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
mu_vi = np.array([-0.82,  2.37, -0.88,  0.37])  # variational means
v_vi  = np.array([ 0.38,  0.19,  0.13,  0.09])  # variational variances (NOT std)
D_vi  = len(mu_vi)
k_param = 1   # which parameter for CI (0-indexed)

# For product-likelihood ELBO term y ~ N(w1*w2, sigma2)
m1_vi, m2_vi = -1.0, 1.0
v1_vi, v2_vi =  1.0, 1.0
y_vi, sigma2_vi = 1.0, 1.0

# ── CALCULATION ──────────────────────────────────────────────────────────────
H_vi = 0.5 * np.sum(np.log(2*np.pi*np.e*v_vi))
print(f"Entropy H[q] = {H_vi:.4f}")
print(f"  Per-dim contributions: {np.round(0.5*np.log(2*np.pi*np.e*v_vi),4)}")
print()

# Credibility intervals
z_vals = {80: 1.282, 90: 1.645, 95: 1.960, 99: 2.576}
print(f"Credibility intervals for mu_{k_param} = {mu_vi[k_param]}, v_{k_param} = {v_vi[k_param]}:")
for pct, z in z_vals.items():
    lo, hi = mu_vi[k_param] - z*np.sqrt(v_vi[k_param]), mu_vi[k_param] + z*np.sqrt(v_vi[k_param])
    print(f"  {pct}% CI: [{lo:.4f}, {hi:.4f}]  (z={z})")
print()

# Expected log-likelihood for product model
E_sq = (y_vi-m1_vi*m2_vi)**2 + m1_vi**2*v2_vi + v1_vi*m2_vi**2 + v1_vi*v2_vi
E_ll = -0.5*np.log(2*np.pi*sigma2_vi) - 0.5/sigma2_vi * E_sq
print(f"E_q[log p(y|w1,w2)] for product model = {E_ll:.4f}")
print(f"  E[(y-w1*w2)^2] = (y-m1*m2)^2 + m1^2*v2 + v1*m2^2 + v1*v2 = {E_sq:.4f}")
print()
print("Mean-field posterior covariance between any two params = 0 (by construction)")


---
## 11. Metropolis MCMC

Replace `log_target_mc` with your log-joint. Everything else stays the same.

Proposal: $\theta^* = \theta^{(k-1)} + \epsilon$, $\epsilon\sim\mathcal{N}(0,\tau^2I)$. Accept if $\log U < \log p(\theta^*)-\log p(\theta^{(k-1)})$.

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
num_iter_mc = 10_000
warmup_frac = 0.1       # fraction to discard as warm-up
tau_mc      = 1.0       # proposal std dev
theta_init_mc = np.array([0.0, 0.0])   # starting point

# For two chains (set use_two_chains=True)
use_two_chains = False
chain_inits_mc = [np.array([-1., 0.]), np.array([2., 5.])]

# ── YOUR LOG-TARGET HERE — replace with your model ───────────────────────────
def log_target_mc(theta):
    # Example: Rosenbrock posterior (2024 exam Part 5)
    z1, z2 = theta
    return -(1-z1)**2 - 20*(z2-z1**2)**2 - z1**2 - z2**2

    # Example: non-linear model y=f(x)+eps, prior w~N(0,I)
    # w1, w2 = theta
    # f_val  = w2 * np.tanh(w1 * x_obs)   # x_obs = your x value
    # return log_npdf(y_obs, f_val, sigma2_mc) + log_npdf(w1,0,1) + log_npdf(w2,0,1)

# ── CALCULATION ──────────────────────────────────────────────────────────────
def run_metropolis(log_target, theta_init, num_iter, tau, seed=42):
    np.random.seed(seed)
    D = len(theta_init)
    samples = [theta_init.copy()]; log_p = log_target(theta_init); n_acc = 0
    for _ in range(num_iter):
        prop = samples[-1] + tau * np.random.randn(D)
        lp   = log_target(prop)
        if np.log(np.random.uniform()) < lp - log_p:
            samples.append(prop); log_p = lp; n_acc += 1
        else:
            samples.append(samples[-1].copy())
    return np.array(samples), n_acc/num_iter

warmup_mc = int(warmup_frac * num_iter_mc)
if not use_two_chains:
    samp_mc, acc_mc = run_metropolis(log_target_mc, theta_init_mc, num_iter_mc, tau_mc)
    posterior_mc = samp_mc[warmup_mc:]
    print(f"Acceptance rate: {acc_mc:.2f}")
else:
    results_mc = [run_metropolis(log_target_mc, init, num_iter_mc, tau_mc, seed=i)
                  for i,init in enumerate(chain_inits_mc)]
    for i,(s,a) in enumerate(results_mc): print(f"Chain {i+1} acc: {a:.2f}")
    posterior_mc = np.vstack([s[warmup_mc:] for s,_ in results_mc])

print(f"Posterior samples: {len(posterior_mc)}")

# Trace plots
fig, axes = plt.subplots(1, posterior_mc.shape[1], figsize=(14, 4))
if posterior_mc.shape[1] == 1: axes = [axes]
for i, ax in enumerate(axes):
    ax.plot(posterior_mc[:, i], lw=0.5)
    ax.set(xlabel='Iteration', ylabel=f'param {i+1}', title=f'Trace param {i+1}')
plt.tight_layout(); plt.show()


---
## 12. Monte Carlo Estimators

All use `posterior_mc` from Section 11. Formula: $\mathbb{E}[g(\theta)]\approx\frac{1}{S}\sum_i g(\theta^{(i)})$, $\;p(\text{event})\approx\frac{1}{S}\sum_i\mathbb{1}[\text{event}]$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
# Uses posterior_mc from Section 11. Or define your own:
# posterior_mc = np.random.multivariate_normal([1,0], [[1,0.5],[0.5,5]], size=6000)

# ── COMMON ESTIMATORS ────────────────────────────────────────────────────────
p1, p2 = posterior_mc[:,0], posterior_mc[:,1]

print(f"E[param1]        = {np.mean(p1):.4f}")
print(f"E[param2]        = {np.mean(p2):.4f}")
print(f"E[(p1-p2)^2]     = {np.mean((p1-p2)**2):.4f}")
print(f"E[sin(p1*p2)]    = {np.mean(np.sin(p1*p2)):.4f}")
print()
print(f"p(param1 > 0)    = {np.mean(p1 > 0):.4f}")
print(f"p(param1 > param2) = {np.mean(p1 > p2):.4f}")
print(f"p(param2 < 1)    = {np.mean(p2 < 1):.4f}")
print()
for pct in [90, 95]:
    t = (100-pct)/2
    print(f"{pct}% CI param1: [{np.percentile(p1,t):.4f}, {np.percentile(p1,100-t):.4f}]")
    print(f"{pct}% CI param2: [{np.percentile(p2,t):.4f}, {np.percentile(p2,100-t):.4f}]")


In [ ]:
# ── Posterior predictive via ancestral sampling ───────────────────────────
# y* = f(x*, w) + eps,   eps ~ N(0, sigma2)
x_star_pp = 2.0
sigma2_pp  = 2.0
f_pp = lambda x, w1, w2: w2 * np.tanh(w1 * x)   # change to your model

np.random.seed(123)
S_pp     = len(posterior_mc)
fstar_pp = f_pp(x_star_pp, posterior_mc[:,0], posterior_mc[:,1])
ystar_pp = fstar_pp + np.sqrt(sigma2_pp)*np.random.randn(S_pp)

print(f"Posterior predictive at x*={x_star_pp}:")
print(f"  E[f*]        = {np.mean(fstar_pp):.4f}")
print(f"  E[y*]        = {np.mean(ystar_pp):.4f}")
print(f"  p(f* > 1)    = {np.mean(fstar_pp > 1):.4f}")
print(f"  95% CI y*:   [{np.percentile(ystar_pp,2.5):.4f}, {np.percentile(ystar_pp,97.5):.4f}]")
print(f"  90% CI y*:   [{np.percentile(ystar_pp,5):.4f}, {np.percentile(ystar_pp,95):.4f}]")


---
## 13. Decision Theory

$\hat{y}^*=\arg\max_{\hat{y}}\mathbb{E}[U(y^*,\hat{y})]$ where $\mathbb{E}[U(y^*,\hat{y}=k)]=\sum_j p(y^*=j)\cdot U(j,k)$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
# Posterior predictive distribution (one entry per class)
p_pred_dt = np.array([1-0.129, 0.129])   # [p(y*=0), p(y*=1)]

# Utility matrix: U[true_class, decision_class]
U_dt = np.array([[2, 1],   # true=0: U(correct)=2, U(wrong)=1
                 [1, 2]])  # true=1: U(wrong)=1,   U(correct)=2

# ── CALCULATION ──────────────────────────────────────────────────────────────
K_dt = len(p_pred_dt)
EU_dt = np.array([np.sum(p_pred_dt * U_dt[:, k]) for k in range(K_dt)])

print("Expected utility per decision:")
for k in range(K_dt):
    terms = " + ".join([f"{p_pred_dt[j]:.3f}*{U_dt[j,k]}" for j in range(K_dt)])
    print(f"  E[U(y*, hat_y={k})] = {terms} = {EU_dt[k]:.4f}")
print(f"Optimal decision: hat_y = {np.argmax(EU_dt)}  (EU = {np.max(EU_dt):.4f})")


---
## 14. Credibility Intervals & Tail Probabilities

**Important:** $v$ in $\mathcal{N}(m,v)$ is **variance** — always take $\sqrt{v}$ to get std before multiplying by $z$.

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
mu_ci   = -0.87     # posterior mean
var_ci  =  0.41     # posterior VARIANCE (not std!)
c_thresh=  0.0      # threshold for tail probability p(w > c)

# ── CALCULATION ──────────────────────────────────────────────────────────────
std_ci = np.sqrt(var_ci)   # ALWAYS sqrt for std
print(f"N({mu_ci}, {var_ci})  ->  std = {std_ci:.4f}")
print()

# All standard intervals
for pct, z in [(80,1.282),(90,1.645),(95,1.960),(99,2.576)]:
    lo = mu_ci - z*std_ci
    hi = mu_ci + z*std_ci
    print(f"  {pct}% CI: [{lo:.4f}, {hi:.4f}]  (z={z})")

# scipy (most reliable for non-standard percentages)
print()
for pct in [80, 90, 95]:
    lo, hi = norm.interval(pct/100, loc=mu_ci, scale=std_ci)
    print(f"  scipy {pct}%: [{lo:.4f}, {hi:.4f}]")

# Tail probabilities
print()
p_tail = 1 - norm.cdf(c_thresh, loc=mu_ci, scale=std_ci)
print(f"p(w > {c_thresh}|y) = 1 - Phi(({c_thresh}-{mu_ci})/std) = {p_tail:.4f}")
print(f"p(w < {c_thresh}|y) = {1-p_tail:.4f}")
print()
print("Prior tail (zero-mean Gaussian): p(w > 0) = 0.5  (exact, by symmetry)")
print("Laplace approx: p(w > w_MAP)    = 0.5  (exact, Gaussian is symmetric around MAP)")


---
## 15. Ancestral Sampling — Prior Mean Estimation

Sample each variable in model order, then average. $\mathbb{E}[y]\approx\frac{1}{S}\sum_i y^{(i)}$

In [ ]:
# ── INPUTS ──────────────────────────────────────────────────────────────────
S_as = 10_000
np.random.seed(0)

# ── EXAMPLE 1: y|w ~ N(e^w, sigma2),  w ~ N(0,1) ────────────────────────────
sigma2_ex1 = 1.0
w_ex1 = np.random.normal(0, 1, S_as)
y_ex1 = np.random.normal(np.exp(w_ex1), np.sqrt(sigma2_ex1))
print(f"E1: y|w ~ N(e^w,{sigma2_ex1}), w~N(0,1):  E[y] = {np.mean(y_ex1):.4f}")
print(f"  Analytical: E[e^w] = exp(0 + 1/2) = {np.exp(0.5):.4f}  (log-normal mean)")

# ── EXAMPLE 2: x|z ~ N(0, e^z), z ~ N(1,1) (2025R exam) ─────────────────────
z_ex2 = np.random.normal(1, 1, S_as)
x_ex2 = np.random.normal(0, np.sqrt(np.exp(z_ex2)))
print(f"E2: x|z ~ N(0,e^z), z~N(1,1):  E[x] = {np.mean(x_ex2):.4f}")
print(f"  p(x < 1) = {np.mean(x_ex2 < 1):.4f}")
print(f"  E[(z-x)^2] = {np.mean((z_ex2-x_ex2)**2):.4f}")

# ── EXAMPLE 3: Poisson GLM mu = exp(3 + w^T x),  w ~ N(0, alpha^-1 I) ───────
alpha_ex3 = 8.0
x_ex3     = 0.0   # test input (at x=0 all weights cancel -> mu = e^3 exact)
w_ex3     = np.random.multivariate_normal([0,0], (1/alpha_ex3)*np.eye(2), S_as)
mu_ex3    = np.exp(3 + w_ex3[:,0]*x_ex3 + w_ex3[:,1]*x_ex3**2)
print(f"E3: Poisson GLM at x*={x_ex3}: E[mu] = {np.mean(mu_ex3):.4f}")
print(f"  Exact at x=0: e^3 = {np.exp(3):.4f}  (weights cancel)")


---
## Quick Reference Table

| Formula | Code |
|---------|------|
| Bayes linear reg posterior $S$ | `inv(alpha*I + (1/sigma^2)*Phi.T@Phi)` |
| Bayes linear reg posterior $m$ | `(1/sigma^2)*S@Phi.T@y` |
| GP posterior mean | `k_s @ solve(C, y)` where `C = K + sigma^2*I` |
| GP posterior variance | `k_ss - k_s @ solve(C, k_s)` |
| GP log marginal likelihood | `mvn.logpdf(y, zeros(N), C)` |
| Linear Gaussian posterior $S^{-1}$ | `inv(Sigma_x) + A.T @ inv(Sigma1) @ A` |
| Linear Gaussian posterior $m$ | `S @ (A.T @ inv(Sigma1) @ (z-b) + inv(Sigma_x) @ mu_x)` |
| Linear Gaussian marginal cov | `Sigma2 + W @ Sigma_z @ W.T` |
| Hessian (logistic regression) | `-X.T @ diag(p*(1-p)) @ X - lam*I` |
| Laplace covariance | `-inv(H)` |
| Probit approximation | `norm.cdf(mu / sqrt(8/pi + var))` |
| Mean-field entropy ($D$ dims) | `0.5*sum(log(2*pi*e*v_i))` |
| Expected utility $k$ | `sum(p_pred * U[:, k])` |

| Interval | $z$-value | Code |
|----------|-----------|------|
| 80% | 1.282 | `norm.interval(0.80, loc=m, scale=sqrt(v))` |
| 90% | 1.645 | `norm.interval(0.90, loc=m, scale=sqrt(v))` |
| 95% | 1.960 | `norm.interval(0.95, loc=m, scale=sqrt(v))` |
| 99% | 2.576 | `norm.interval(0.99, loc=m, scale=sqrt(v))` |

**Common gotchas:**
- $S$ matrix diagonal entries are **variances** → take $\sqrt{S_{ii}}$ for std before CI
- GP prior variance at $x^*$: `k_ss = kappa**2 * exp(0) = kappa**2`  
- Prior predictive is symmetric → `p(y*=0|x*) = 0.5` for zero-mean GP  
- Laplace is symmetric → `p(w > w_MAP) = 0.5` always  
- Prior is symmetric → `p(w > 0) = 0.5` for zero-mean Gaussian prior  
- Mean-field covariance between any two params = **0** always  
- Metropolis accept in log-space: `log(U) < log_p_prop - log_p_cur`